# Calculate IST (Instantaneous Shot Threat)

## Imports

In [1]:
import numpy as np
import pandas as pd

In [2]:
import sys
from pathlib import Path
# Add project root to path for imports
ROOT = Path.cwd().parent  
sys.path.append(str(ROOT))

from src.data_io.maps import load_maps_npz
from src.viz.court import plot_player_map_on_court
from src.utils.players import maps_npz_player_dict
from src.features.ist.features import compute_ist_row, add_ist_column
from src.pipelines.defensive_features import run_season_pipeline

## Load Data

In [3]:
maps, meta = load_maps_npz("../data/processed/shot_maps/maps_1ft_xfg.npz")
defense_df = pd.read_parquet("../data/processed/def_features/defense_features.parquet")

In [4]:
print(defense_df.shape)
defense_df.columns

(108, 31)


Index(['GAME_ID', 'SHOT_EVENT_ID', 'tracking_event_id', 'release_frame_idx',
       'event_list_idx', 'PERIOD', 'game_clock', 'PLAYER_ID', 'TEAM_ID',
       'x_ft', 'y_ft', 'xFG_offense', 'xPPS_offense', 'SHOT_MADE_FLAG',
       'close_def_dist_release', 'closest_def_dist', 'close_def_id',
       'num_defenders_tracked', 'w0_close_def_dist_mean',
       'w0_close_def_dist_min', 'w0_shooter_speed_mean',
       'w0_shooter_speed_max', 'w0_def_speed_mean', 'w0_closing_speed_mean',
       'w1_close_def_dist_mean', 'w1_close_def_dist_min',
       'w1_shooter_speed_mean', 'w1_shooter_speed_max', 'w1_def_speed_mean',
       'w1_closing_speed_mean', 'shooter_speed'],
      dtype='object')

In [5]:
pid2row = {int(p): i for i, p in enumerate(maps["player_ids"])}
dist_to_ball = 0.0
row = defense_df.iloc[0]

## Calculating IST

In [6]:
shots_with_ist = add_ist_column(defense_df, maps, pid2row)
shots_with_ist[["PLAYER_ID","x_ft","y_ft","IST","IST_Q","IST_O","IST_S","SHOT_MADE_FLAG"]].head()

,PLAYER_ID,x_ft,y_ft,IST,IST_Q,IST_O,IST_S,SHOT_MADE_FLAG
0,201567,-2.083333,18.250000,0.058631,0.366944,0.998665,0.159997,0
1,202691,14.000000,14.833333,0.039175,0.364379,0.196360,0.547521,1
2,2544,-4.333333,20.333333,0.125753,0.361904,0.999948,0.347494,0
3,203110,-11.250000,17.833333,0.044400,0.351155,0.236085,0.535574,0
4,101106,1.833333,4.250000,0.189258,0.429889,0.992544,0.443556,0


In [7]:
shots_with_ist.sort_values("IST", ascending=False).head(10)[
    ["PLAYER_ID","x_ft","y_ft","IST","IST_Q","IST_O","IST_S","SHOT_MADE_FLAG"]
]


,PLAYER_ID,x_ft,y_ft,IST,IST_Q,IST_O,IST_S,SHOT_MADE_FLAG
107,203099,1.166667,20.750000,0.361907,0.450000,0.999594,0.804566,0
71,2571,-0.333333,2.333333,0.349330,0.519268,0.918660,0.732299,0
18,203521,-1.583333,5.166667,0.324937,0.404072,0.875863,0.918131,0
54,2544,-1.416667,20.583333,0.324790,0.362634,0.999904,0.895727,0
55,202691,-17.666667,13.250000,0.316268,0.352827,0.960284,0.933457,0
75,201939,5.250000,20.750000,0.315070,0.363055,0.906994,0.956819,1
76,203521,-9.916667,18.583333,0.311756,0.359482,0.999789,0.867418,1
89,203521,-9.333333,18.666667,0.293567,0.359482,0.999970,0.816663,1
94,2592,11.750000,6.666667,0.293177,0.450000,0.999993,0.651509,1
106,203546,9.583333,20.166667,0.292802,0.356386,0.997332,0.823786,0


In [8]:
shots_with_ist.groupby("SHOT_MADE_FLAG")[["IST", "IST_Q", "IST_O", "IST_S"]].mean()

,IST,IST_Q,IST_O,IST_S
SHOT_MADE_FLAG,,,,
0,0.150097,0.437810,0.802003,0.473652
1,0.148608,0.468936,0.849390,0.427435


In [9]:
shots_with_ist[["IST", "IST_Q", "IST_O", "IST_S"]].describe()

,IST,IST_Q,IST_O,IST_S
count,108.000000,108.000000,108.000000,108.000000
mean,0.149380,0.452797,0.824819,0.451399
std,0.085993,0.123305,0.281530,0.240051
min,0.002161,0.300097,0.012517,0.043421
25%,0.084615,0.366163,0.749094,0.253277
50%,0.139046,0.405026,0.976528,0.393441
75%,0.201209,0.458849,0.999790,0.621733
max,0.361907,0.797487,1.000000,0.956819


Loading shots from: ../data/processed/shots/all_season_shots.parquet
Found 1 files to process in ../data/raw/7z_test/
[1/1] ERROR on 01.01.2016.CHA.at.TOR.7z: 'SevenZipFile' object has no attribute 'readall'

Pipeline Complete.
